# Chest X-ray 肺炎分類 — EfficientNet_B3 遷移學習（本機）

模組都在 repo 的 `.py` 裡，這本 notebook 只負責把它們串起來。
Colab 版在 `efficient_train_colab.ipynb`，流程一致，差別只在多了 clone / Drive / push。

| Cell | 做什麼 |
| --- | --- |
| 1 | 匯入模組、偵測裝置 |
| 2 | **實驗設定（改參數只改這裡）** |
| 3 | 下載資料與清點 |
| 4 | 原始切分的類別分佈 |
| 5 | 重新切分 70/15/15 |
| 6 | 準備資料集與模型 |
| 7 | 訓練 |
| 8 | 測試集評估 |
| 9 | 訓練曲線與混淆矩陣 |
| 10 | 儲存結果 |

Kaggle 憑證放 `~/.kaggle/kaggle.json`，或設 `KAGGLE_USERNAME` / `KAGGLE_KEY` 環境變數。

In [ ]:
# Cell 1 — 匯入模組與裝置
import sys

sys.path.insert(0, "..")
%matplotlib inline

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from constant import DEFAULT_MODEL_NAME, POSITIVE_CLASS_NAME
from data.dataset import KAGGLE_DATASET_ID, dataset_version, download_dataset, make_dataloaders
from data.split import (
    build_datasets_from_plan,
    collect_all_images,
    count_images,
    group_counts,
    patient_group_key,
    plan_splits,
    split_counts,
)
from engine.trainer import compute_class_weights, run_one_epoch, train_model
from engine.visualize import plot_class_distribution, plot_cm, plot_history
from env import collect_run_info
from models.builder import build_model
from utils import EpochTimer, collect_config, make_output_dir, make_run_id, save_results, set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

run_info = collect_run_info()
print(f"使用裝置: {device}")
print(f"torch {run_info['torch_version']}")
print(f"程式碼版本 git_commit: {run_info['git_commit']}")

In [ ]:
# Cell 2 — 實驗設定（只改這個 cell）
EXPERIMENT_NAME     = "e1_b3_frozen"
MODEL_NAME          = DEFAULT_MODEL_NAME
IMAGE_SIZE          = 224
BATCH_SIZE          = 32
EPOCHS              = 30
LR                  = 1e-3
WEIGHT_DECAY        = 1e-4
NUM_WORKERS         = 0

RANDOM_SEED         = 43     # 只影響初始化與批次順序
SPLIT_SEED          = 42     # 只影響資料切分


TRAIN_RATIO         = 0.70
VAL_RATIO           = 0.15
TEST_RATIO          = 0.15

# 同一位病人整組進同一個 split
GROUP_BY_PATIENT    = True

# 訓練策略：E1 骨幹全程凍結
FREEZE_BACKBONE     = True   # 凍結骨幹，只訓練分類頭
UNFREEZE_EPOCH      = None   # None = 全程不解凍
BACKBONE_LR_FACTOR  = 0.1    # 只有解凍後才用得到
PATIENCE            = 6      # early stopping 的耐心值

# 路徑
# Windows 上 num_workers 用 0 較穩
OUTPUT_BASE         = "../outputs"
DATASET_LOCAL_DIR   = None


# 收集設定進 metadata.json
CONFIG = collect_config(globals())

print(f"實驗: {EXPERIMENT_NAME}")
print(f"RANDOM_SEED={RANDOM_SEED}（訓練）  SPLIT_SEED={SPLIT_SEED}（切分）")

In [ ]:
# Cell 3 — 下載資料與清點
raw_root = download_dataset(local_dir=DATASET_LOCAL_DIR)
print(f"資料集路徑: {raw_root}\n")

counts = count_images(raw_root)
for split, per_class in counts.items():
    print(f"{split:<5} {per_class}")

In [ ]:
# Cell 4 — 原始切分的類別分佈
# 官方 val 只有 16 張，太少
plot_class_distribution(counts, title="Original Kaggle Split", show=True)

In [ ]:
# Cell 5 — 重新切分 70/15/15
# 索引式切分，不建實體資料夾
all_images = collect_all_images(raw_root)
print("總張數:", {c: len(v) for c, v in all_images.items()})

group_key = patient_group_key if GROUP_BY_PATIENT else None
if group_key is not None:
    # 群組數等於張數表示沒分組
    print("群組數:", group_counts(all_images, group_key))

plan = plan_splits(
    all_images,
    split_seed=SPLIT_SEED,          # 參數名是 split_seed
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    group_key=group_key,            # 整組進同一個 split
)

counts_after = split_counts(plan)
for split, per_class in counts_after.items():
    print(f"{split:<5} {per_class}  (共 {sum(per_class.values())})")

plot_class_distribution(counts_after, title="Regrouped 70/15/15", show=True)

split_info = {
    "split_scheme": "regrouped_70_15_15"
    + ("_grouped_by_patient" if GROUP_BY_PATIENT else ""),
    "split_seed": SPLIT_SEED,
    "split_ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "split_counts": counts_after,
    "dataset_id": KAGGLE_DATASET_ID,
    "dataset_version": dataset_version(raw_root),
}

In [ ]:
# Cell 6 — 準備資料集與模型
# set_seed 放這裡才可重現
set_seed(RANDOM_SEED)

image_datasets, class_to_idx, idx_to_class = build_datasets_from_plan(plan, img_size=IMAGE_SIZE)
num_classes = len(class_to_idx)
positive_index = class_to_idx.get(POSITIVE_CLASS_NAME, 1)
print(f"類別: {class_to_idx}, 正類別索引({POSITIVE_CLASS_NAME}): {positive_index}")

dataloaders = make_dataloaders(image_datasets, BATCH_SIZE, num_workers=NUM_WORKERS)

model = build_model(
    num_classes, device,
    model_name=MODEL_NAME,
    freeze_backbone=FREEZE_BACKBONE,
)

class_weights = compute_class_weights(image_datasets["train"], num_classes, device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
print(f"模型與資料集準備完成，類別數：{num_classes}")

In [ ]:
# Cell 7 — 訓練
timer = EpochTimer()

best_state, best_val_f1, best_cm, best_epoch, history = train_model(
    model, device, criterion, optimizer, scheduler, dataloaders,
    epochs=EPOCHS,
    unfreeze_epoch=UNFREEZE_EPOCH,
    lr=LR,
    backbone_lr_factor=BACKBONE_LR_FACTOR,
    should_freeze_backbone=FREEZE_BACKBONE,
    positive_index=positive_index,
    epoch_callback=timer,
    early_stopping=True,
    patience=PATIENCE,
)
print(f"\n訓練完成，最佳驗證 F1：{best_val_f1:.4f}（Epoch {best_epoch + 1}/{len(history)}）")

time_per_epoch = timer.get_time_per_epoch()

In [ ]:
# Cell 8 — 測試集評估
model.load_state_dict(best_state)
test_m = run_one_epoch(model, device, dataloaders["test"], criterion, positive_index=positive_index)

print(
    "\n[Test] "
    f"loss={test_m['loss']:.4f}, acc={test_m['accuracy']:.4f}, "
    f"precision={test_m['precision']:.4f}, recall={test_m['recall']:.4f}, f1={test_m['f1']:.4f}"
)

In [ ]:
# Cell 9 — 訓練曲線與混淆矩陣
plot_history(history, show=True)
plot_cm(test_m["cm"], class_names=[idx_to_class[0], idx_to_class[1]], show=True)

In [ ]:
# Cell 10 — 儲存結果
run_id = make_run_id(EXPERIMENT_NAME)
out_dir = make_output_dir(EXPERIMENT_NAME, base=OUTPUT_BASE, run_id=run_id)

save_results(
    out_dir, model, best_state, best_val_f1, best_epoch, test_m,
    class_to_idx, idx_to_class, IMAGE_SIZE, history,
    model_name=MODEL_NAME, train_time_per_epoch=time_per_epoch,
    config=CONFIG, run_info=run_info, split_info=split_info,
)
print(f"\nrun_id: {run_id}")